In [1]:
import os
import re
import sys
import json
import pandas as pd
sys.path.append('../')
import numpy as np
from pyts.transformation import ROCKET
from sklearn.linear_model import RidgeClassifierCV
from transform import TimeSeriesTransform

In [2]:
INSTANCES_DIR = '../data/linear_acuator/instances/'
INFERENCE_DIR = '../data/linear_acuator/inference/'
STATES = ['normal', 
            'backlash1', 'backlash2',
            'lackLubrication1', 'lackLubrication2',
            'spalling1', 'spalling2', 'spalling3', 'spalling4', 'spalling5', 'spalling6', 'spalling7', 'spalling8']
LOADS= ['20kg', '40kg', '-40kg']

In [3]:
def get_X_y(data_dir, filenames, load):
    cfg = json.load(open("../config/config.json"))
    ts_trans = TimeSeriesTransform(cfg)
    
    X, y = [], []
    for filename in filenames:
        load_num = load[:-2]
        state = re.match(fr'(.*)_{load_num}', filename).group(1)
        df = pd.read_csv(os.path.join(data_dir, load, state, filename))
        tmp_cur = ts_trans.smoothing(ts_df=df, field='current')
        # tmp_pos = ts_transform.smoothing(ts_df=df, field='position_error')
        X.append(tmp_cur)
        y.append(state)
    return np.array(X), np.array(y)

In [4]:
load = '20kg'
filenames_20kg = [os.listdir(os.path.join(INSTANCES_DIR, load, state)) for state in STATES]
filenames_20kg = [filename for sublist in filenames_20kg for filename in sublist]

load = '40kg'
filenames_40kg = [os.listdir(os.path.join(INSTANCES_DIR, load, state)) for state in STATES]
filenames_40kg = [filename for sublist in filenames_40kg for filename in sublist]

load = '-40kg'
filenames_m40kg = [os.listdir(os.path.join(INSTANCES_DIR, load, state)) for state in STATES]
filenames_m40kg = [filename for sublist in filenames_m40kg for filename in sublist]

In [5]:
load = '20kg'
filenames_20kg_t = [os.listdir(os.path.join(INFERENCE_DIR, load, state)) for state in STATES]
filenames_20kg_t = [filename for sublist in filenames_20kg_t for filename in sublist]

load = '40kg'
filenames_40kg_t = [os.listdir(os.path.join(INFERENCE_DIR, load, state)) for state in STATES]
filenames_40kg_t = [filename for sublist in filenames_40kg_t for filename in sublist]

load = '-40kg'
filenames_m40kg_t = [os.listdir(os.path.join(INFERENCE_DIR, load, state)) for state in STATES]
filenames_m40kg_t = [filename for sublist in filenames_m40kg_t for filename in sublist]

In [6]:
X_20kg, y_20kg = get_X_y(INSTANCES_DIR, filenames_20kg, load='20kg')
X_40kg, y_40kg = get_X_y(INSTANCES_DIR, filenames_40kg, load='40kg')
X_m40kg, y_m40kg = get_X_y(INSTANCES_DIR, filenames_m40kg, load='-40kg')

In [7]:
X_20kg_t, y_20kg_t = get_X_y(INFERENCE_DIR, filenames_20kg_t, load='20kg')
X_40kg_t, y_40kg_t = get_X_y(INFERENCE_DIR, filenames_40kg_t, load='40kg')
X_m40kg_t, y_m40kg_t = get_X_y(INFERENCE_DIR, filenames_m40kg_t, load='-40kg')

In [8]:
X_20kg.shape, X_20kg_t.shape

((520, 324), (130, 324))

## Use pyts.transformation.ROCKET (Only on the CPU)

In [9]:
rocket = ROCKET(kernel_sizes=[11], n_kernels=4900, random_state=0)

rocket.fit(X_20kg)

rocket_results_20kg = rocket.transform(X_20kg)
rocket_results_20kg_t = rocket.transform(X_20kg_t)
rocket_20 = RidgeClassifierCV()
rocket_20.fit(rocket_results_20kg, y_20kg)
print(f'20kg train Ridge: {rocket_20.score(rocket_results_20kg, y_20kg)}')
print(f'20kg test Ridge: {rocket_20.score(rocket_results_20kg_t, y_20kg_t)}')

20kg train Ridge: 1.0
20kg test Ridge: 0.8153846153846154


In [10]:
rocket.fit(X_40kg)

rocket_results_40kg = rocket.transform(X_40kg)
rocket_results_40kg_t = rocket.transform(X_40kg_t)
rocket_40 = RidgeClassifierCV()
rocket_40.fit(rocket_results_40kg, y_40kg)
print(f'40kg train: {rocket_40.score(rocket_results_40kg, y_40kg)}')
print(f'40kg test: {rocket_40.score(rocket_results_40kg_t, y_40kg_t)}')

40kg train: 1.0
40kg test: 0.8372093023255814


In [11]:
rocket.fit(X_m40kg)

rocket_results_m40kg = rocket.transform(X_m40kg)
rocket_results_m40kg_t = rocket.transform(X_m40kg_t)
rocket_m40 = RidgeClassifierCV()
rocket_m40.fit(rocket_results_m40kg, y_m40kg)
print(f'm40kg train: {rocket_m40.score(rocket_results_m40kg, y_m40kg)}')
print(f'-40kg test: {rocket_m40.score(rocket_results_m40kg_t, y_m40kg_t)}')

m40kg train: 1.0
-40kg test: 0.8307692307692308


In [204]:
from time import perf_counter

time = []
for _ in range(5):
    time_a = perf_counter()
    r = rocket.transform(X_20kg)
    rocket_20.fit(r, y_20kg)
    time_b = perf_counter()
    time.append(time_b - time_a)
print(f'Time taken: {time_b - time_a}')

Time taken: 5.456165399984457


In [17]:
time = []
for _ in range(5):
    time_a = perf_counter()
    r = rocket.transform(X_20kg_t)
    rocket_20.score(r, y_20kg_t)
    time_b = perf_counter()
    time.append(time_b - time_a)
print(f'Time taken: {time_b - time_a}')

Time taken: 1.3065765000064857


## ROCKET support for the GPU acceleration

In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [26]:
# y_20kg_labels = np.concatenate([np.where(y==np.unique(y_20kg)) for y in y_20kg]).squeeze()
# y_20kg_t_labels = np.concatenate([np.where(y==np.unique(y_20kg)) for y in y_20kg_t]).squeeze()

# train_dataset = TensorDataset(
#     torch.tensor(X_20kg).float().unsqueeze(1).to(device),
#     torch.tensor(y_20kg_labels).long().to(device),
# )

# test_dataset = TensorDataset(
#     torch.tensor(X_20kg_t).float().unsqueeze(1).to(device),
#     torch.tensor(y_20kg_t_labels).long().to(device),
# )

# train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [13]:
from rocket import PyTorchROCKET, PytorchRidgeClassifier, train, predict

In [14]:
rocket_torch = PyTorchROCKET(kernel_sizes=[9, 11], n_kernels=1000, random_state=0, device='cuda')

In [37]:
rocket_torch.fit(X_20kg)
train_features = rocket_torch.transform(X_20kg)

ridge = RidgeClassifierCV()
ridge.fit(train_features.detach().cpu().numpy(), y_20kg)
test_features = rocket_torch.transform(X_20kg_t)
print(f'{ridge.score(test_features.cpu().numpy(), y_20kg_t)}')

0.7846153846153846


In [26]:
rocket_torch.fit(X_40kg)
train_features = rocket_torch.transform(X_40kg)

ridge = RidgeClassifierCV()
ridge.fit(train_features.detach().cpu().numpy(), y_40kg)
test_features = rocket_torch.transform(X_40kg_t)
print(f'{ridge.score(test_features.cpu().numpy(), y_40kg_t)}')

0.8062015503875969


In [27]:
rocket_torch.fit(X_m40kg)
train_features = rocket_torch.transform(X_m40kg)

ridge = RidgeClassifierCV()
ridge.fit(train_features.detach().cpu().numpy(), y_m40kg)
test_features = rocket_torch.transform(X_m40kg_t)
print(f'{ridge.score(test_features.cpu().numpy(), y_m40kg_t)}')

0.8076923076923077


In [42]:
# Get the total parameters number
rocket_params = rocket_torch.weights_.shape[0] * rocket_torch.weights_.shape[1] + rocket_torch.bias_.shape[0] + rocket_torch.dilation_.shape[0] + rocket_torch.padding_.shape[0]
ridge_params = ridge.coef_.shape[0] * ridge.coef_.shape[1] + ridge.intercept_.shape[0]
print(f'Rocket params: {rocket_params}')
print(f'Ridge params: {ridge_params}')
print(f'total params: {rocket_params + ridge_params}')

Rocket params: 14000
Ridge params: 26013
total params: 40013


In [38]:
from time import perf_counter

time = []
for _ in range(5):
    time_a = perf_counter()
    
    rocket_torch.fit(X_20kg)
    train_features = rocket_torch.transform(X_20kg)
    ridge.fit(train_features.detach().cpu().numpy(), y_20kg)

    time_b = perf_counter()
    time.append(time_b - time_a)
print(f'Average Time taken: {np.array(time).mean()}')

Average Time taken: 0.9252117600175552


In [39]:
time = []
for _ in range(5):
    time_a = perf_counter()
    
    test_features = rocket_torch.transform(X_m40kg_t)
    ridge.score(test_features.cpu().numpy(), y_m40kg_t)

    time_b = perf_counter()
    time.append(time_b - time_a)
print(f'Average Time taken: {np.array(time).mean()}')

Average Time taken: 0.36915184003300966


In [ ]:

test_features = rocket_torch.transform(X_20kg_t)
ridge_torch = PytorchRidgeClassifier(input_dim=train_features.shape[1], num_classes=len(STATES))
ridge_torch.to(device=device)
y_20kg_labels = np.concatenate([np.where(y==np.unique(y_20kg)) for y in y_20kg]).squeeze()
y_20kg_t_labels = np.concatenate([np.where(y==np.unique(y_20kg)) for y in y_20kg_t]).squeeze()

# train_dataset = TensorDataset(
#     torch.tensor(X_20kg).float().unsqueeze(1).to(device),
#     torch.tensor(y_20kg_labels).long().to(device),
# )

# test_dataset = TensorDataset(
#     torch.tensor(X_20kg_t).float().unsqueeze(1).to(device),
#     torch.tensor(y_20kg_t_labels).long().to(device),
# )

# train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

ridge_torch = train(ridge_torch, 
      train_features, 
      torch.tensor(y_20kg_labels).long().to(device), 
      l2_reg=1.0, epochs=100, lr=0.01)
predict(ridge_torch, test_features)

In [34]:
# import pickle

# with open('rocket_torch.pkl', 'wb') as f:
#     pickle.dump(rocket_torch, f)
# with open('ridge.pkl', 'wb') as f:
#     pickle.dump(ridge, f)

torch.save(ridge_torch.state_dict(), '../outputs/ridge_torch.pt')
torch.save(rocket_torch.weights_, '../outputs/rocket_torch_weights.pt')
torch.save(rocket_torch.bias_, '../outputs/rocket_torch_bias.pt')
torch.save(rocket_torch.dilation_, '../outputs/rocket_torch_dilation.pt')
torch.save(rocket_torch.padding_, '../outputs/rocket_torch_padding.pt')
torch.save(rocket_torch.length_, '../outputs/rocket_torch_length.pt')